<h2> Imports, loading event-log function and cleaning pipeline </h2>

In [11]:
import numpy as np
import pandas as pd
import pm4py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import statistics
from collections import Counter
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from collections import defaultdict
from sklearn.model_selection import GridSearchCV
import itertools
from ACF_code import acf_matrices, pmi


In [28]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'subtrace': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'subtrace': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, False)
    test_df = build_prefix_df(test_split, prefix_lengths, False)

    return train_df, test_df, train_split, test_split


<h1> Loading event-logs and transforming</h1>

<h4> Loading datasets </h4>

In [29]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
#GLOBAL_prefix_lengths = [100, 150, 200, 300, 400, 500, 600, 700, 800]

train_df_2013, test_df_2013, full_train_2013, full_test_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 2718.17it/s]


In [4]:
print(train_df_2013.head())
print(full_train_2013.head())

    resource                                           subtrace  \
0      Aline  [Accepted, Accepted, Accepted, Completed, Acce...   
1  Krzysztof  [Accepted, Accepted, Accepted, Queued, Accepte...   
2    Yangeun  [Accepted, Accepted, Completed, Accepted, Acce...   
3     Lukasz  [Queued, Completed, Accepted, Queued, Accepted...   
4    Chethan  [Accepted, Queued, Accepted, Queued, Accepted,...   

   prefix_length last_activity next_activity  
0             10        Queued     Completed  
1             10      Accepted     Completed  
2             10      Accepted        Queued  
3             10     Completed     Completed  
4             10        Queued     Completed  
org:resource
Jan-Ivar                              [Queued, Queued, Queued]
Sofia C      [Accepted, Accepted, Accepted, Accepted, Accep...
Toshiyuki                             [Queued, Queued, Queued]
Aline        [Accepted, Queued, Accepted, Accepted, Queued,...
Antonio      [Accepted, Queued, Accepted, Complete

In [10]:
def train_RF(X_train, X_test, y_train, y_test):
    rf = RandomForestClassifier(n_estimators=100, random_state=1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy is: {accuracy}")
    f1score = f1_score(y_test, y_pred, average='weighted')
    print(f"f1score = {f1score}")

    report = classification_report(y_test, y_pred, output_dict=True)
    print(report)
    return rf

In [19]:
# Fitting one hot encoder on the training data and applying on both train and test sets
def ohe_data(train_df, test_df, feature_cols=['last_activity']):
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    encoder.fit(train_df[feature_cols])

    X_train = encoder.transform(train_df[feature_cols])
    X_test = encoder.transform(test_df[feature_cols])

    cols_before = train_df[feature_cols].shape[1]
    cols_after = X_train.shape[1]
    expansian_factor = cols_after / cols_before if cols_before > 0 else 0

    print(f"Prefix length (input columns): {cols_before}")    
    print(f"One-hot columns (output features): {cols_after}")
    print(f"Expansion factor = {expansian_factor}")

    return X_train, X_test, encoder

In [ ]:
# Training random forest on un-bucketed 
print("Global OHE performance:")
X_train, X_test, ohe_encoder = ohe_data(train_df_2013, test_df_2013)
global_ohe_rf = train_RF(X_train, X_test, train_df_2013['next_activity'], test_df_2013['next_activity']) 


Global OHE performance:
273719
273719
59045
Accuracy is: 0.593174697264798
f1score = 0.536184662777818


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Accepted': {'precision': 0.6634254729288976, 'recall': 0.8189069498263801, 'f1-score': 0.7330120047748823, 'support': 39742.0}, 'Completed': {'precision': 0.24817299028931825, 'recall': 0.23861776879391663, 'f1-score': 0.24330159976445187, 'support': 10389.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8914.0}, 'accuracy': 0.593174697264798, 'macro avg': {'precision': 0.30386615440607195, 'recall': 0.3525082395400989, 'f1-score': 0.32543786817977804, 'support': 59045.0}, 'weighted avg': {'precision': 0.4902044938818863, 'recall': 0.593174697264798, 'f1-score': 0.536184662777818, 'support': 59045.0}}


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [14]:
def transform_subtrace_to_columns(df, length):
    # This turns the list [A, B, C] into columns: pos_1, pos_2, pos_3
    # with values A, B, C
    prefix_cols = pd.DataFrame(df['subtrace'].tolist(), index=df.index)
    prefix_cols.columns = [f'pos_{i}' for i in range(length)]
    return prefix_cols

In [20]:
param_grid = {
    'n_estimators': [50, 100,200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'bootstrap': [True, False]
}

results = []
best_acc = 0
best_length = 0
best_params = {}


for length in GLOBAL_prefix_lengths:
    print(f"Searching for length {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    if current_train_df.empty or current_test_df.empty:
        print(f"Skipping length {length}: Insufficient data.")
        continue

    # --- NEW STEP: Explode the subtrace list into separate position columns ---
    X_train_exploded = transform_subtrace_to_columns(current_train_df, length)
    X_test_exploded = transform_subtrace_to_columns(current_test_df, length)
    feature_cols = X_train_exploded.columns.tolist()



    current_X_train, current_X_test, current_encoder = ohe_data(X_train_exploded, X_test_exploded, feature_cols=feature_cols)

    current_y_train = current_train_df['next_activity']
    current_y_test = current_test_df['next_activity']

    # rf = train_RF(current_X_train, current_X_test, current_y_train, current_y_test)
    rf_base = RandomForestClassifier(random_state=1)

    grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, 
                               cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    
    grid_search.fit(current_X_train, current_y_train)
    
    best_model = grid_search.best_estimator_
    current_y_pred = best_model.predict(current_X_test)
    
    acc = accuracy_score(current_y_test, current_y_pred)
    f1 = f1_score(current_y_test, current_y_pred, average='weighted')

    print(f"Accuracy for length {length} is {acc}")

    if acc > best_acc:
        best_acc = acc
        best_length = length
        best_params = grid_search.best_params_


print(f"Best accuracy found is {best_acc} for length {best_length} with RF params {best_params}")

    

Searching for length 10
Prefix length (input columns): 10
One-hot columns (output features): 32
Expansion factor = 3.2
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Accuracy for length 10 is 0.7916666666666666
Searching for length 20
Prefix length (input columns): 20
One-hot columns (output features): 61
Expansion factor = 3.05
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Accuracy for length 20 is 0.7889908256880734
Searching for length 30
Prefix length (input columns): 30
One-hot columns (output features): 91
Expansion factor = 3.033333333333333
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Accuracy for length 30 is 0.7790697674418605
Searching for length 40
Prefix length (input columns): 40
One-hot columns (output features): 121
Expansion factor = 3.025
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Accuracy for length 40 is 0.8378378378378378
Searching for length 50
Prefix length (input columns): 50
One-hot columns (out

<h3> Creating last-state encoding </h3>

<h3> Creating bigrams </h3>

In [13]:
def create_bigram_features(df_train, df_test):

    # all activities in the train set
    train_activities = set([act for prefix in df_train['subtrace'] for act in prefix])
    unique_activities = sorted(list(train_activities))

    # all possible bigram transitions
    all_transitions = [f"{a}->{b}" for a in unique_activities for b in unique_activities]

    #print(f"Found {len(unique_activities)} unique activities.")
    print(f"Creating features for {len(all_transitions)} possible bigrams...")

    def _extract_counts(df):
        bigram_rows = []
        for prefix in df['subtrace']:
            counts = defaultdict(int)
            
            for i in range(len(prefix) - 1):
                key = f"{prefix[i]}->{prefix[i+1]}"
                counts[key] += 1
            
            row = [counts[t] for t in all_transitions]
            bigram_rows.append(row)
            
        return pd.DataFrame(bigram_rows, columns=all_transitions, index=df.index)

    X_train_bigram = _extract_counts(df_train)
    X_test_bigram = _extract_counts(df_test)
    
    return X_train_bigram, X_test_bigram

X_train2, X_test2 = create_bigram_features(train_df_2013, test_df_2013)
y_train2 = train_df_2013['next_activity']
y_test2 = test_df_2013['next_activity']

# Non-bucketed random forest on bigram encoded 
train_RF(X_train2, X_test2, y_train2, y_test2)

best_acc = 0
best_length = 0
best_params = {}

param_grid = {
    'n_estimators': [50, 100,200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'bootstrap': [True, False]
}

results = []




# Bucketed random forest on bigram encoded
for length in GLOBAL_prefix_lengths:
    print(f"Searching for length {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    current_X_train, current_X_test = create_bigram_features(current_train_df, current_test_df)

    current_y_train = current_train_df['next_activity']
    current_y_test = current_test_df['next_activity']

    rf = train_RF(current_X_train, current_X_test, current_y_train, current_y_test)

    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                               cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    
    grid_search.fit(current_X_train, current_y_train)
    
    best_model = grid_search.best_estimator_
    current_y_pred = best_model.predict(current_X_test)
    
    acc = accuracy_score(current_y_test, current_y_pred)
    f1 = f1_score(current_y_test, current_y_pred, average='weighted')



    if acc > best_acc:
        print(f"New best accuracy found for length: {length} and it is {acc}")
        best_acc = acc
        best_length = length
        best_params = grid_search.best_params_



print(f"Best accuracy: {best_acc} for length {best_length}")
print(f" With best RF params: ")
    





Creating features for 16 possible bigrams...
Accuracy is: 0.791869918699187
f1score = 0.7867346908760161
{'Accepted': {'precision': 0.4, 'recall': 0.2916666666666667, 'f1-score': 0.3373493975903614, 'support': 48.0}, 'Completed': {'precision': 0.8742393509127789, 'recall': 0.8941908713692946, 'f1-score': 0.884102564102564, 'support': 482.0}, 'Queued': {'precision': 0.4827586206896552, 'recall': 0.49411764705882355, 'f1-score': 0.4883720930232558, 'support': 85.0}, 'accuracy': 0.791869918699187, 'macro avg': {'precision': 0.5856659905341447, 'recall': 0.5599917283649283, 'f1-score': 0.5699413515720604, 'support': 615.0}, 'weighted avg': {'precision': 0.7831184551196425, 'recall': 0.791869918699187, 'f1-score': 0.7867346908760161, 'support': 615.0}}
Searching for length 10
Creating features for 16 possible bigrams...
Accuracy is: 0.75
f1score = 0.7516426282051282
{'Accepted': {'precision': 0.3, 'recall': 0.3, 'f1-score': 0.3, 'support': 10.0}, 'Completed': {'precision': 0.844660194174757

In [26]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Train word2vec
word2vec_model = Word2Vec(sentences=train_df_2013['subtrace'], vector_size=16, window=5, min_count=1, sg=1)

def word2vec_embed_data(w2v_model, df, window_size=3):
    X = []
    y = df['next_activity'].values
    for trace in df['subtrace']:
        current_window = trace[-window_size:]

        vectors = []
        for act in current_window:
            if act in w2v_model.wv:
                vectors.append(w2v_model.wv[act])
            else:
                vectors.append(np.zeros(w2v_model.vector_size))
        
        while len(vectors) < window_size:
            vectors.insert(0, np.zeros(w2v_model.vector_size))
        X.append(vectors)
    X = np.array(X)
    X_flat = X.reshape(X.shape[0], -1)

    return X_flat, y

all_possible_activities = pd.concat([
    train_df_2013['next_activity'], 
    test_df_2013['next_activity']
]).unique()

le = LabelEncoder()
le.fit(all_possible_activities) 

param_grid = {
    'n_estimators': [50, 100,200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'bootstrap': [True, False]
}


import itertools

# Define Word2Vec specific hyperparams to test
w2v_tuning_grid = {
    'vector_size': [16, 32],
    'window': [3, 5],
    'sg': [1] # 1 for skip-gram
}

# Generate combinations for Word2Vec
w2v_combinations = [dict(zip(w2v_tuning_grid.keys(), v)) 
                    for v in itertools.product(*w2v_tuning_grid.values())]

best_acc = 0
best_results = {}

for length in GLOBAL_prefix_lengths:
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]
    
    if current_train_df.empty or current_test_df.empty:
        continue

    for w2v_p in w2v_combinations:
        # 1. Train Word2Vec on the specific training subtraces for this length
        model = Word2Vec(
            sentences=current_train_df['subtrace'],
            vector_size=w2v_p['vector_size'],
            window=w2v_p['window'],
            min_count=1,
            sg=w2v_p['sg'],
            seed=1
        )

        # 2. Embed the data using the sliding window logic
        current_X_train, current_y_train_raw = word2vec_embed_data(model, current_train_df, w2v_p['window'])
        current_X_test, current_y_test_raw = word2vec_embed_data(model, current_test_df, w2v_p['window'])

        # 3. Use Global LabelEncoder
        current_y_train_encoded = le.transform(current_y_train_raw)
        
        # 4. GridSearch for Random Forest
        rf = RandomForestClassifier(random_state=1)
        grid_search = GridSearchCV(
            estimator=rf,
            param_grid=param_grid, # Your RF grid
            cv=5,
            scoring='accuracy',
            n_jobs=-1, # Use all cores
            verbose=0
        )

        grid_search.fit(current_X_train, current_y_train_encoded)
        
        # 5. Evaluate
        best_rf = grid_search.best_estimator_
        y_pred_encoded = best_rf.predict(current_X_test)
        
        # Inverse transform to compare with raw strings or just use encoded comparison
        acc = accuracy_score(le.transform(current_y_test_raw), y_pred_encoded)
        
        print(f"Len: {length} | W2V: {w2v_p} | RF Best: {grid_search.best_params_} | Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            best_results = {
                'length': length,
                'w2v_params': w2v_p,
                'rf_params': grid_search.best_params_
            }

print("-" * 30)
print(f"BEST ACCURACY: {best_acc:.4f}")
print(f"BEST PARAMS: {best_params}")
print(f"Best prefix length: {best_length}")




Len: 10 | W2V: {'vector_size': 16, 'window': 3, 'sg': 1} | RF Best: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100} | Acc: 0.7708
Len: 10 | W2V: {'vector_size': 16, 'window': 5, 'sg': 1} | RF Best: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100} | Acc: 0.7083
Len: 10 | W2V: {'vector_size': 32, 'window': 3, 'sg': 1} | RF Best: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100} | Acc: 0.7708
Len: 10 | W2V: {'vector_size': 32, 'window': 5, 'sg': 1} | RF Best: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200} | Acc: 0.7292
Len: 20 | W2V: {'vector_size': 16, 'window': 3, 'sg': 1} | RF Best: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50} | Acc: 0.7248
Len: 20 | W2V: {'vector_size': 16, 'window': 5, 'sg': 1} | RF Best: {'bootstrap': True, 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100} | Acc: 0.6606
Len: 

In [ ]:
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

le = LabelEncoder()
all_activities = pd.concat([train_df_2013['next_activity'], test_df_2013['next_activity']])
le.fit(all_activities)

def prepare_tagged_data_from_series(series):
    tagged_docs = []
    for resource_id, trace_list in series.items():
        tagged_docs.append(TaggedDocument(words=trace_list, tags=[str(resource_id)]))
    return tagged_docs

print("Tagging document...")
full_train_tagged = prepare_tagged_data_from_series(full_train_2013)
print(f"Done tagging. Created {len(full_train_tagged)} documents.")

print("Training Global Doc2Vec Model.")
model = Doc2Vec(
    vector_size=64,
    window=5,
    min_count=1,
    workers=4,
    epochs=40  
)

model.build_vocab(x)
model.train(full_train_tagged, total_examples=model.corpus_count, epochs=model.epochs)
print("Global Model Trained")


best_accuracy = 0
best_length = 0

for length in GLOBAL_prefix_lengths:
    print(f"Processing for length: {length}")

    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    print(f"Current training size: {len(current_train_df)}")
    print(f"Current testing size: {len(current_test_df)}")

    if current_train_df.empty or current_test_df.empty:
        print(f"Skipping length {length} (insufficient data)")
        continue

    X_train = np.array([
        model.infer_vector(s, epochs=50) for s in current_train_df['subtrace']
    ])
    
    X_test = np.array([
        model.infer_vector(s, epochs=50) for s in current_test_df['subtrace']
    ])
    
    y_train = le.transform(current_train_df['next_activity'])
    y_test = le.transform(current_test_df['next_activity'])

    rf = RandomForestClassifier(n_estimators=100, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"Accuracy for length {length}: {acc:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_length = length

print(f"\nFinal Result: Best Accuracy {best_accuracy:.4f} found at length {best_length}")

Tagging document...
Done tagging. Created 1025 documents.
Training Global Doc2Vec Model.
Global Model Trained
Processing for length: 10
Current training size: 552
Current testing size: 144
Accuracy for length 10: 0.8194
Processing for length: 20
Current training size: 389
Current testing size: 109
Accuracy for length 20: 0.7706
Processing for length: 30
Current training size: 310
Current testing size: 86
Accuracy for length 30: 0.8256
Processing for length: 40
Current training size: 263
Current testing size: 74
Accuracy for length 40: 0.8243
Processing for length: 50
Current training size: 231
Current testing size: 60
Accuracy for length 50: 0.8333
Processing for length: 75
Current training size: 178
Current testing size: 44
Accuracy for length 75: 0.8636
Processing for length: 100
Current training size: 138
Current testing size: 40
Accuracy for length 100: 0.8000
Processing for length: 125
Current training size: 115
Current testing size: 32
Accuracy for length 125: 0.7812
Processing f

In [ ]:
print(best_accuracy)
print(best_length)

"""
Tagging document...
Done tagging. Created 1025 documents.
Training Global Doc2Vec Model.
Global Model Trained
Processing for length: 10
Current training size: 552
Current testing size: 144
Accuracy for length 10: 0.8194
Processing for length: 20
Current training size: 389
Current testing size: 109
Accuracy for length 20: 0.7706
Processing for length: 30
Current training size: 310
Current testing size: 86
Accuracy for length 30: 0.8256
Processing for length: 40
Current training size: 263
Current testing size: 74
Accuracy for length 40: 0.8243
Processing for length: 50
Current training size: 231
Current testing size: 60
Accuracy for length 50: 0.8333
Processing for length: 75
Current training size: 178
Current testing size: 44
Accuracy for length 75: 0.8636
Processing for length: 100
Current training size: 138
Current testing size: 40
Accuracy for length 100: 0.8000
Processing for length: 125
Current training size: 115
Current testing size: 32
Accuracy for length 125: 0.7812
Processing for length: 150
Current training size: 92
Current testing size: 26
Accuracy for length 150: 0.8462

Final Result: Best Accuracy 0.8636 found at length 75

--- Processing length 200 ---
Accuracy for length 200: 0.6918

--- Processing length 300 ---
Accuracy for length 300: 0.6667

--- Processing length 400 ---
Accuracy for length 400: 0.6520

--- Processing length 500 ---
Accuracy for length 500: 0.6400

--- Processing length 600 ---
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Accuracy for length 600: 0.6615

--- Processing length 700 ---
Accuracy for length 700: 0.6267

--- Processing length 800 ---
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Accuracy for length 800: 0.6000

Final Result: Best Accuracy 0.6918 found at length 200
"""

0.6917808219178082
200


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertConfig, BertForMaskedLM, BertModel
from torch.optim import AdamW
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# setting up BERT vocabulary
all_activities = sorted(list(set([act for seq in full_train_2013 for act in seq])))
vocab = {act: i + 5 for i, act in enumerate(all_activities)} 
vocab['[PAD]'] = 0
vocab['[MASK]'] = 1
vocab['[CLS]'] = 2
vocab['[SEP]'] = 3
vocab['[UNK]'] = 4
inv_vocab = {v: k for k, v in vocab.items()}

def tokenize(seq, max_len):
    seq_tokens = [vocab['[CLS]']] + [vocab.get(act, vocab['[UNK]']) for act in seq[-max_len+1:]]
    
    pad_len = max_len - len(seq_tokens)
    input_ids = seq_tokens + [vocab['[PAD]']] * pad_len
    
    # create attention mask (1 for real data, 0 for pads)
    attention_mask = [1] * len(seq_tokens) + [0] * pad_len
    
    return torch.tensor(input_ids), torch.tensor(attention_mask)


class LogDataset(Dataset):
    def __init__(self, traces, max_len=150):
        self.traces = traces
        self.max_len = max_len
        
    def __len__(self):
        return len(self.traces)
    
    def __getitem__(self, idx):
        return tokenize(self.traces[idx], self.max_len)

# configuring BERT
config = BertConfig(
    vocab_size=len(vocab),
    hidden_size=128, 
    num_hidden_layers=4, 
    num_attention_heads=4,
    intermediate_size=512
)

# pre-training
def pretrain_bert(full_traces, epochs=5, batch_size=32):
    model = BertForMaskedLM(config)
    model.train()
    optimizer = AdamW(model.parameters(), lr=1e-4)
    
    dataset = LogDataset(full_traces)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    print("Starting Pre-training...")
    for epoch in range(epochs):
        total_loss = 0
        for input_ids, attention_mask in loader:
            labels = input_ids.clone()
        
            rand = torch.rand(input_ids.shape)
            
            mask_arr = (rand < 0.15) & (input_ids > 4)
            input_ids[mask_arr] = vocab['[MASK]']
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")
        
    return model.bert 

bert_encoder = pretrain_bert(full_train_2013)

def get_embeddings(df, encoder, max_len=150):
    encoder.eval()
    embeddings = []
    
    # We can also batch this for speed, but row-by-row is okay for inference if dataset is small
    with torch.no_grad():
        for _, row in df.iterrows():
            input_ids, attention_mask = tokenize(row['subtrace'], max_len)
            
            input_ids = input_ids.unsqueeze(0)
            attention_mask = attention_mask.unsqueeze(0)
            
            outputs = encoder(input_ids, attention_mask=attention_mask)
            
            last_real_idx = attention_mask.sum() - 1 
            
            emb = outputs.last_hidden_state[0, last_real_idx, :].numpy()
            embeddings.append(emb)
            
    return np.array(embeddings)


Starting Pre-training...


/var/folders/gc/9ml3w_r92xn0tsh3p0k69h8m0000gn/T/ipykernel_14981/1175761288.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return tokenize(self.traces[idx], self.max_len)


Epoch 1/5 - Loss: 1.6128
Epoch 2/5 - Loss: 0.9807
Epoch 3/5 - Loss: 0.6187
Epoch 4/5 - Loss: 0.3898
Epoch 5/5 - Loss: 0.2756


In [33]:
best_acc = 0
best_len = 0

for length in GLOBAL_prefix_lengths:
    print(f"Running BERT for length: {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    print(f"Train set for this length is: {len(current_train_df)}")
    print(f"test set for this length is: {len(current_test_df)}")

    if current_train_df.empty or current_test_df.empty:
        print(f"Skipping length {length} (insufficient data)")
        continue

    X_train = get_embeddings(current_train_df, bert_encoder)
    X_test = get_embeddings(current_test_df, bert_encoder)

    label_map = {act: i for i, act in enumerate(all_activities)}
    y_train = np.array([label_map[a] for a in current_train_df['next_activity']])
    y_test = np.array([label_map.get(a, -1) for a in current_test_df['next_activity']]) # Handle unseen labels in test

    rf = RandomForestClassifier(n_estimators=100, random_state=1)
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    current_acc = accuracy_score(y_test, preds)
    print(f"Random Forest Accuracy: {accuracy_score(y_test, preds):.4f}")
    if best_acc < current_acc:
        best_acc = current_acc
        best_len = length

print(f"Best accuracy was: {best_acc} for length {best_len}")

Running BERT for length: 10
Train set for this length is: 552
test set for this length is: 144
Random Forest Accuracy: 0.7292
Running BERT for length: 20
Train set for this length is: 389
test set for this length is: 109
Random Forest Accuracy: 0.7339
Running BERT for length: 30
Train set for this length is: 310
test set for this length is: 86
Random Forest Accuracy: 0.7442
Running BERT for length: 40
Train set for this length is: 263
test set for this length is: 74
Random Forest Accuracy: 0.7703
Running BERT for length: 50
Train set for this length is: 231
test set for this length is: 60
Random Forest Accuracy: 0.7500
Running BERT for length: 75
Train set for this length is: 178
test set for this length is: 44
Random Forest Accuracy: 0.7727
Running BERT for length: 100
Train set for this length is: 138
test set for this length is: 40
Random Forest Accuracy: 0.8000
Running BERT for length: 125
Train set for this length is: 115
test set for this length is: 32
Random Forest Accuracy: 0.8

In [ ]:
"""
Running BERT for length: 10
Random Forest Accuracy: 0.7292
Running BERT for length: 20
Random Forest Accuracy: 0.7523
Running BERT for length: 30
Random Forest Accuracy: 0.7442
Running BERT for length: 40
Random Forest Accuracy: 0.7838
Running BERT for length: 50
Random Forest Accuracy: 0.8500
Running BERT for length: 75
Random Forest Accuracy: 0.8182
Running BERT for length: 100
Random Forest Accuracy: 0.7500
Running BERT for length: 125
Random Forest Accuracy: 0.6875
Running BERT for length: 150
Random Forest Accuracy: 0.7692



Running BERT for length: 100
Random Forest Accuracy: 0.6374
Running BERT for length: 150
Random Forest Accuracy: 0.6368
Running BERT for length: 200
Random Forest Accuracy: 0.6549
Running BERT for length: 300
Random Forest Accuracy: 0.6245
Running BERT for length: 400
Random Forest Accuracy: 0.5591
Running BERT for length: 500
Random Forest Accuracy: 0.6400
Running BERT for length: 600
Random Forest Accuracy: 0.6369
Running BERT for length: 700
Random Forest Accuracy: 0.5156
Running BERT for length: 800
Random Forest Accuracy: 0.5920


"""

Running BERT for length: 10
Random Forest Accuracy: 0.7292
Running BERT for length: 20
Random Forest Accuracy: 0.7523
Running BERT for length: 30
Random Forest Accuracy: 0.7442
Running BERT for length: 40
Random Forest Accuracy: 0.7838
Running BERT for length: 50
Random Forest Accuracy: 0.8500
Running BERT for length: 75
Random Forest Accuracy: 0.8182
Running BERT for length: 100
Random Forest Accuracy: 0.7500
Running BERT for length: 125
Random Forest Accuracy: 0.6875
Running BERT for length: 150
Random Forest Accuracy: 0.7692


In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score



NGRAM_SIZE = 5      
BAG_OF_WORDS = 1    
PPMI = 1            

# converting list of lists to list for ACF functions
train_log = full_train_2013.values.tolist()

# creating alphabet from log
alphabet = set(act for trace in train_log for act in trace)

print(f"generating matrices for {len(alphabet)} activities")


# creating raw ACF matrix
dist_mat, acf_embeddings, act_freq, ctx_freq, ctx_idx = acf_matrices.get_activity_context_frequency_matrix(
    log=train_log, 
    alphabet=alphabet, 
    ngram_size=NGRAM_SIZE, 
    bag_of_words=BAG_OF_WORDS
)

# applying pmi post-processing
_, pmi_embeddings = pmi.get_activity_context_frequency_matrix_pmi(
    embeddings=acf_embeddings,
    activity_freq_dict=act_freq,
    context_freq_dict=ctx_freq,
    context_index=ctx_idx,
    ppmi=PPMI
)

print("embeddings generated successfully")

# creating applying the embeddings
def vectorize_sequences(sequences, embeddings, method='average'):
    """
    Converts a list of activity sequences (prefixes) into a matrix of feature vectors.
    
    Args:
        sequences: List of lists (the 'subtrace' column)
        embeddings: Dict mapping activity -> numpy vector
        method: 'last' (embedding of last act) or 'average' (mean of all acts in window)
    """
    feature_matrix = []
    embedding_size = len(next(iter(embeddings.values())))
    
    for seq in sequences:
        # filtering out activities that might not be in our training alphabet (handling unseen data)
        valid_acts = [act for act in seq if act in embeddings]
        
        if not valid_acts:
            # if we have not seen the activity in train, place all 0's
            feature_matrix.append(np.zeros(embedding_size))
            continue

        # This creates an embedding for just the last activity in a trace, what usually happens after this specific activity          
        if method == 'last':
            vec = embeddings[valid_acts[-1]]
            
        # This averages the embedding of an entire subtrace to create a more detailed 'context'    
        elif method == 'average':
            vectors = [embeddings[act] for act in valid_acts]
            vec = np.mean(vectors, axis=0)
            
        feature_matrix.append(vec)
        
    return np.array(feature_matrix)

best_acc = 0
best_length = 0

for length in GLOBAL_prefix_lengths:
    print(f"Now running for prefix length: {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    print(f"Train set for this length is: {len(current_train_df)}")
    print(f"test set for this length is: {len(current_test_df)}")

    # transforming training data
    X_train = vectorize_sequences(current_train_df['subtrace'], pmi_embeddings, method='average')
    y_train = current_train_df['next_activity']


    # using same encodings learned from train to encode test data
    X_test = vectorize_sequences(current_test_df['subtrace'], pmi_embeddings, method='average')
    y_test = current_test_df['next_activity']


    print("Training Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    print("Predicting...")
    y_pred = rf.predict(X_test)

    current_acc = accuracy_score(y_test, y_pred)


    # Evaluation
    print("Accuracy:", current_acc)
    if current_acc > best_acc:
        best_acc = current_acc
        best_length = length

print(f"Best accuracy found: {best_acc} for length: {length}")


generating matrices for 4 activities
embeddings generated successfully
Now running for prefix length: 10
Train set for this length is: 45155
test set for this length is: 10584
Training Random Forest...
Predicting...
Accuracy: 0.6704459561602418
Now running for prefix length: 20
Train set for this length is: 40515
test set for this length is: 9327
Training Random Forest...
Predicting...
Accuracy: 0.6682749008255602
Now running for prefix length: 30
Train set for this length is: 36996
test set for this length is: 8358
Training Random Forest...
Predicting...
Accuracy: 0.6679827709978464
Now running for prefix length: 40
Train set for this length is: 34110
test set for this length is: 7536
Training Random Forest...
Predicting...
Accuracy: 0.6669320594479831
Now running for prefix length: 50
Train set for this length is: 31627
test set for this length is: 6853
Training Random Forest...
Predicting...
Accuracy: 0.671968480957245
Now running for prefix length: 75
Train set for this length is: 

In [ ]:
"""
ACF/PMI:
Length: 20
Accuracy: 0.668


Length: 30
Accuracy: 0.668


Length: 40
Accuracy: 0.667


Length: 50
Accuracy: 0.672


Length: 75
Accuracy: 0.663


Length: 100
Accuracy: 0.659


Length: 125
Accuracy: 0.632


Length: 150
Accuracy: 0.612
"""

' \nACF/PMI:\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\nLength: \nAccuracy: \n\n\n\n\n\n'